# RQ1: Performance Analysis

**Research Question:** How do latency (P50, P99) and throughput differ across Monolithic, Microservices, and Triton architectures under varying load?

## Hypotheses Tested

| ID | Statement | Testable Prediction |
|---|---|---|
| H1a | Monolithic exhibits lowest P99 latency at low concurrency (<=10 users) | monolithic.p99 < microservices.p99 AND monolithic.p99 < triton.p99 |
| H1b | Microservices P99 latency is competitive with monolithic | (microservices.p99 - monolithic.p99) / monolithic.p99 < 0.20 |
| H1c | Triton shows lower latency variance at high concurrency (>=50 users) | (triton.p99 - triton.p50) < (microservices.p99 - microservices.p50) |
| H1d | All architectures reach saturation (P99 > 500ms) before 100 users | all(arch.p99 > 500) for concurrent_users < 100 |

In [ ]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from analysis.utilities.loaders import ResultsLoader

# Load configuration from experiment.yaml
ResultsLoader._load_config_from_yaml()

# Get saturation threshold from experiment.yaml (H1d hypothesis)
SATURATION_THRESHOLD_MS = ResultsLoader.SATURATION_THRESHOLD_MS

# Set publication-quality defaults
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

# Output directory for plots
PLOTS_DIR = Path('../plots/rq1')
PLOTS_DIR.mkdir(exist_ok=True, parents=True)


def cohens_d(group1, group2):
    """Calculate Cohen's d effect size between two groups.
    
    Cohen's d measures the standardized difference between two means.
    Interpretation: |d| < 0.2 = negligible, 0.2-0.5 = small, 0.5-0.8 = medium, > 0.8 = large
    """
    n1, n2 = len(group1), len(group2)
    var1, var2 = group1.var(), group2.var()
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
    return (group1.mean() - group2.mean()) / pooled_std if pooled_std > 0 else 0


def interpret_cohens_d(d):
    """Interpret Cohen's d effect size magnitude."""
    d_abs = abs(d)
    if d_abs < 0.2:
        return 'negligible'
    elif d_abs < 0.5:
        return 'small'
    elif d_abs < 0.8:
        return 'medium'
    else:
        return 'large'


print(f"Saturation threshold (from experiment.yaml H1d): {SATURATION_THRESHOLD_MS}ms")

In [ ]:
# Load data
loader = ResultsLoader()
df = loader.load_summary()
agg = loader.load_aggregate()

# Display data overview
print(f"Total runs: {len(df)}")
print(f"Architectures: {df['architecture'].unique()}")
print(f"User levels: {sorted(df['concurrent_users'].unique())}")
print(f"\nSample data:")
df.head()

In [ ]:
# Data Validation
# Cross-check computed values against expected baseline

# Basic sanity checks
assert df['throughput_rps'].min() > 0, "Negative throughput detected"
assert df['client_p99_ms'].min() > 0, "Negative latency detected"
assert not df['client_p99_ms'].isna().any(), "NaN values in latency"
assert df['error_rate_percent'].max() <= 100, "Error rate > 100%"

# Validate expected ranges (from prior runs)
# P99 at 100 users should be in saturation range (>500ms)
p99_at_100 = df[df['concurrent_users'] == 100].groupby('architecture')['client_p99_ms'].mean()
for arch, p99 in p99_at_100.items():
    assert p99 > 100, f"Unexpectedly low P99 for {arch} at 100 users: {p99:.0f}ms"

# Validate run count consistency
runs_per_config = df.groupby(['architecture', 'concurrent_users']).size()
assert runs_per_config.min() == 3, "Some configurations have fewer than 3 runs"
assert runs_per_config.max() == 3, "Some configurations have more than 3 runs"

print("Data validation passed")
print(f"  - All values positive and within expected ranges")
print(f"  - Each configuration has exactly 3 runs")
print(f"  - P99 at 100 users: {p99_at_100.to_dict()}")

## 1. Throughput vs. Concurrent Users

Shows how each architecture's throughput scales with increasing load. Key for identifying saturation points (H1d).

In [ ]:
# Aggregate throughput by architecture and user count
throughput_agg = df.groupby(['architecture', 'concurrent_users'])['throughput_rps'].agg(['mean', 'std']).reset_index()

fig, ax = plt.subplots(figsize=(10, 6))

for arch in loader.ARCH_DISPLAY_NAMES.keys():
    arch_data = throughput_agg[throughput_agg['architecture'] == arch]
    ax.errorbar(
        arch_data['concurrent_users'],
        arch_data['mean'],
        yerr=arch_data['std'],
        label=loader.ARCH_DISPLAY_NAMES[arch],
        color=loader.ARCH_COLORS[arch],
        marker='o',
        capsize=3,
        linewidth=2,
        markersize=6
    )

ax.set_xlabel('Concurrent Users')
ax.set_ylabel('Throughput (requests/second)')
ax.set_title('Throughput Scaling by Architecture')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
ax.set_xticks(loader.USER_LEVELS)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'rq1_throughput_vs_users.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: rq1_throughput_vs_users.png")

## 2. P99 Latency vs. Concurrent Users (Log Scale)

Critical visualization for H1a, H1b. Log scale reveals differences at low load and shows saturation behavior.

In [ ]:
# Aggregate P99 latency
latency_agg = df.groupby(['architecture', 'concurrent_users'])['client_p99_ms'].agg(['mean', 'std']).reset_index()

fig, ax = plt.subplots(figsize=(10, 6))

for arch in loader.ARCH_DISPLAY_NAMES.keys():
    arch_data = latency_agg[latency_agg['architecture'] == arch]
    ax.errorbar(
        arch_data['concurrent_users'],
        arch_data['mean'],
        yerr=arch_data['std'],
        label=loader.ARCH_DISPLAY_NAMES[arch],
        color=loader.ARCH_COLORS[arch],
        marker='o',
        capsize=3,
        linewidth=2,
        markersize=6
    )

# Add saturation threshold line (H1d: P99 > threshold) - loaded from experiment.yaml
ax.axhline(y=SATURATION_THRESHOLD_MS, color='gray', linestyle='--', alpha=0.7, 
           label=f'Saturation threshold ({SATURATION_THRESHOLD_MS}ms)')

ax.set_xlabel('Concurrent Users')
ax.set_ylabel('P99 Latency (ms) - Log Scale')
ax.set_title('P99 Latency Scaling by Architecture')
ax.set_yscale('log')
ax.legend(loc='best')
ax.grid(True, alpha=0.3, which='both')
ax.set_xticks(loader.USER_LEVELS)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'rq1_p99_latency_vs_users_log.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: rq1_p99_latency_vs_users_log.png")

## 3. Latency Percentile Comparison (P50 vs P99)

Visualizes latency variance (P99 - P50 gap) for H1c testing.

In [ ]:
# Compute latency variance metrics
df_variance = loader.get_latency_variance(df)
variance_agg = df_variance.groupby(['architecture', 'concurrent_users'])['p99_p50_gap_ms'].agg(['mean', 'std']).reset_index()

fig, ax = plt.subplots(figsize=(10, 6))

for arch in loader.ARCH_DISPLAY_NAMES.keys():
    arch_data = variance_agg[variance_agg['architecture'] == arch]
    ax.errorbar(
        arch_data['concurrent_users'],
        arch_data['mean'],
        yerr=arch_data['std'],
        label=loader.ARCH_DISPLAY_NAMES[arch],
        color=loader.ARCH_COLORS[arch],
        marker='s',
        capsize=3,
        linewidth=2,
        markersize=6
    )

ax.set_xlabel('Concurrent Users')
ax.set_ylabel('Latency Variance (P99 - P50) in ms')
ax.set_title('Latency Variance by Architecture (H1c)')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
ax.set_xticks(loader.USER_LEVELS)

# Highlight high concurrency region (H1c condition: >=50 users)
ax.axvspan(50, 100, alpha=0.1, color='yellow', label='H1c region (>=50 users)')

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'rq1_latency_variance.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: rq1_latency_variance.png")

## 4. Hypothesis Validation

### H1a: Monolithic lowest P99 at low concurrency (<=10 users)

In [ ]:
# H1a: Test at low concurrency (users <= 10)
low_load = df[df['concurrent_users'] <= 10]

h1a_results = low_load.groupby('architecture')['client_p99_ms'].agg(['mean', 'std', 'count'])
print("H1a: P99 Latency at Low Concurrency (<=10 users)")
print("="*60)
print(h1a_results.sort_values('mean'))
print()

mono_p99 = h1a_results.loc['monolithic', 'mean']
micro_p99 = h1a_results.loc['microservices', 'mean']
triton_p99 = h1a_results.loc['triton', 'mean']

h1a_supported = mono_p99 < micro_p99 and mono_p99 < triton_p99
print(f"Prediction: monolithic.p99 ({mono_p99:.1f}ms) < microservices.p99 ({micro_p99:.1f}ms) AND monolithic.p99 < triton.p99 ({triton_p99:.1f}ms)")
print(f"H1a Supported: {h1a_supported}")

# Effect size calculations for H1a
mono_data = low_load[low_load['architecture'] == 'monolithic']['client_p99_ms']
micro_data = low_load[low_load['architecture'] == 'microservices']['client_p99_ms']
triton_data = low_load[low_load['architecture'] == 'triton']['client_p99_ms']

# Cohen's d: positive means first group has higher values
d_mono_vs_micro = cohens_d(mono_data, micro_data)
d_mono_vs_triton = cohens_d(mono_data, triton_data)

print(f"\nEffect Sizes (Cohen's d):")
print(f"  Monolithic vs Microservices: d = {d_mono_vs_micro:.2f} ({interpret_cohens_d(d_mono_vs_micro)})")
print(f"  Monolithic vs Triton: d = {d_mono_vs_triton:.2f} ({interpret_cohens_d(d_mono_vs_triton)})")

# Store for summary table
h1a_d = min(abs(d_mono_vs_micro), abs(d_mono_vs_triton))  # Conservative: smallest effect

In [ ]:
# H1b: Microservices competitive with monolithic (overhead < 20%)
print("\nH1b: Microservices Overhead vs Monolithic (<=10 users)")
print("="*60)

overhead_pct = (micro_p99 - mono_p99) / mono_p99
h1b_tolerance = 0.20
h1b_supported = overhead_pct < h1b_tolerance

print(f"Monolithic P99: {mono_p99:.1f}ms")
print(f"Microservices P99: {micro_p99:.1f}ms")
print(f"Overhead: {overhead_pct*100:.1f}%")
print(f"Tolerance: {h1b_tolerance*100:.0f}%")
print(f"H1b Supported: {h1b_supported}")

# Effect size for H1b (same comparison as H1a mono vs micro)
h1b_d = abs(d_mono_vs_micro)
print(f"\nEffect Size (Cohen's d): d = {h1b_d:.2f} ({interpret_cohens_d(h1b_d)})")

In [ ]:
# H1c: Triton lower variance at high concurrency (>=50 users)
high_load = df_variance[df_variance['concurrent_users'] >= 50]

h1c_results = high_load.groupby('architecture')['p99_p50_gap_ms'].agg(['mean', 'std'])
print("\nH1c: Latency Variance at High Concurrency (>=50 users)")
print("="*60)
print(h1c_results.sort_values('mean'))
print()

triton_var = h1c_results.loc['triton', 'mean']
micro_var = h1c_results.loc['microservices', 'mean']

h1c_supported = triton_var < micro_var
print(f"Prediction: triton.variance ({triton_var:.1f}ms) < microservices.variance ({micro_var:.1f}ms)")
print(f"H1c Supported: {h1c_supported}")

# Effect size for H1c (variance comparison)
triton_var_data = high_load[high_load['architecture'] == 'triton']['p99_p50_gap_ms']
micro_var_data = high_load[high_load['architecture'] == 'microservices']['p99_p50_gap_ms']

h1c_d = cohens_d(triton_var_data, micro_var_data)
print(f"\nEffect Size (Cohen's d): d = {h1c_d:.2f} ({interpret_cohens_d(h1c_d)})")

In [ ]:
# H1d: Saturation before 100 users (P99 > threshold)
print(f"\nH1d: Saturation Analysis (P99 > {SATURATION_THRESHOLD_MS}ms threshold)")
print("="*60)

# Use saturation threshold from experiment.yaml (loaded in cell-1)
saturation_threshold = SATURATION_THRESHOLD_MS

saturation_points = {}
for arch in ['monolithic', 'microservices', 'triton']:
    arch_data = df[df['architecture'] == arch].groupby('concurrent_users')['client_p99_ms'].mean()
    
    # Find first user level where P99 > threshold
    saturated_at = arch_data[arch_data > saturation_threshold]
    if len(saturated_at) > 0:
        saturation_point = saturated_at.index[0]
        saturation_points[arch] = saturation_point
        print(f"{arch}: Saturated at {saturation_point} users (P99={arch_data[saturation_point]:.0f}ms)")
    else:
        saturation_points[arch] = None
        print(f"{arch}: Not saturated within test range")

# H1d: All architectures must saturate BEFORE reaching 100 users
# This means saturation point must be < 100 for all architectures
h1d_supported = all(
    point is not None and point < 100 
    for point in saturation_points.values()
)

print(f"\nSaturation points: {saturation_points}")
print(f"H1d Supported (all saturate before 100 users): {h1d_supported}")

# No Cohen's d for H1d - it's a threshold test, not a comparison
h1d_d = None

## 5. Summary Statistics Table

Performance metrics with standard deviations at representative load levels (1, 10, 50, 100 users).

In [ ]:
# Generate summary table for thesis with SD columns
# Use 4 representative load levels per CONTEXT.md
representative_levels = [1, 10, 50, 100]
df_repr = df[df['concurrent_users'].isin(representative_levels)]

# Aggregate with mean and std for key metrics
summary_agg = df_repr.groupby(['architecture', 'concurrent_users']).agg({
    'throughput_rps': ['mean', 'std'],
    'client_p50_ms': ['mean', 'std'],
    'client_p99_ms': ['mean', 'std'],
    'error_rate_percent': 'mean'
})

# Flatten column names
summary_agg.columns = [
    'Throughput', 'Throughput SD',
    'P50 (ms)', 'P50 SD',
    'P99 (ms)', 'P99 SD',
    'Error Rate (%)'
]

# Round for presentation
summary_table = summary_agg.round(2)

print("\nRQ1 Summary Table (Representative Load Levels)")
print("="*80)
print(summary_table.to_string())

# Export to CSV for thesis
summary_table.to_csv(PLOTS_DIR / 'rq1_summary_table.csv')
print("\nSaved: rq1_summary_table.csv")

## 6. Hypothesis Results Summary

### Effect Size Interpretation

Cohen's d effect sizes indicate the practical significance of observed differences:

| Magnitude | Cohen's d | Interpretation |
|-----------|-----------|----------------|
| Negligible | |d| < 0.2 | Difference exists but practically insignificant |
| Small | 0.2 <= |d| < 0.5 | Noticeable with careful measurement |
| Medium | 0.5 <= |d| < 0.8 | Observable difference in practice |
| Large | |d| >= 0.8 | Substantial, easily observable difference |

**Note:** With n=3 runs per configuration, statistical power is limited. Effect sizes should be interpreted as indicative rather than definitive.

In [ ]:
# Comprehensive hypothesis results table with effect sizes
hypothesis_results = pd.DataFrame([
    {
        'Hypothesis': 'H1a',
        'Statement': 'Monolithic lowest P99 at low load (<=10 users)',
        'Predicted': 'mono < micro AND mono < triton',
        'Observed': f'{mono_p99:.0f}ms vs {micro_p99:.0f}ms, {triton_p99:.0f}ms',
        'Supported': h1a_supported,
        'Cohen_d': f'{h1a_d:.2f}',
        'Interpretation': interpret_cohens_d(h1a_d)
    },
    {
        'Hypothesis': 'H1b',
        'Statement': 'Microservices overhead < 20%',
        'Predicted': '(micro - mono) / mono < 0.20',
        'Observed': f'{overhead_pct*100:.1f}% overhead',
        'Supported': h1b_supported,
        'Cohen_d': f'{h1b_d:.2f}',
        'Interpretation': interpret_cohens_d(h1b_d)
    },
    {
        'Hypothesis': 'H1c',
        'Statement': 'Triton lower variance at high load (>=50 users)',
        'Predicted': 'triton.var < micro.var',
        'Observed': f'{triton_var:.0f}ms vs {micro_var:.0f}ms',
        'Supported': h1c_supported,
        'Cohen_d': f'{h1c_d:.2f}',
        'Interpretation': interpret_cohens_d(h1c_d)
    },
    {
        'Hypothesis': 'H1d',
        'Statement': f'All saturate before 100 users (P99 > {SATURATION_THRESHOLD_MS}ms)',
        'Predicted': 'saturation_point < 100 for all',
        'Observed': ', '.join([f"{k}: {v}" for k, v in saturation_points.items()]),
        'Supported': h1d_supported,
        'Cohen_d': 'N/A',
        'Interpretation': 'threshold test'
    },
])

print("\nRQ1 Hypothesis Results")
print("="*100)
print(hypothesis_results.to_string(index=False))

# Save to CSV
hypothesis_results.to_csv(PLOTS_DIR / 'rq1_hypothesis_results.csv', index=False)
print("\nSaved: rq1_hypothesis_results.csv")

# Summary counts
supported_count = sum(1 for s in [h1a_supported, h1b_supported, h1c_supported, h1d_supported] if s is True)
print(f"\nSummary: {supported_count}/4 hypotheses supported")